In [ ]:
import pandas as pd
import numpy as np
import sumolib
from modules.sumo import TLIncomingCorridors,GetRouteGreenTime,create_initial_tl_df,get_upper_bounds
import torch
from modules.inverse_cdf import InverseCDF

In [2]:
all_routes_df=None

route_files_list=["./Ingolstadt Network/InTAS_016.rou.xml","./Ingolstadt Network/InTAS_017.rou.xml","./Ingolstadt Network/InTAS_018.rou.xml"]
for route_file in route_files_list:
    route_df=pd.read_xml(route_file,iterparse={"route":["edges"]})
    route_df=route_df.rename(columns={"edges":"route_edges"})
    if all_routes_df is None:
        all_routes_df=route_df
    else:
        all_routes_df=pd.concat([all_routes_df,route_df],axis=0,ignore_index=True)

In [3]:
additional_gt_xml="./Ingolstadt Network/additional_gt.xml"
net_xml="./Ingolstadt Network/ingolstadt.net.xml"


corridors_df=torch.load("./corridors_savings/corridors_df.pt",weights_only=False)
complete_corridors_df=torch.load("./corridors_savings/complete_corridors_connexions_df.pt",weights_only=False)

initial_tl_df=create_initial_tl_df(additional_gt_xml)

route_green_time_accessor=GetRouteGreenTime(net_file=net_xml,initial_tl_df=initial_tl_df)

In [13]:
corridors_df[["tl","route_str_str","route_id","route_last_edge","route_demand"]].iloc[217]

tl                                        cluster_30503255_335525526
route_str_str      29238188#0 -25665374 29027727#6 25665375#0 320...
route_id                                                         492
route_last_edge                                             54858974
route_demand                                                       3
Name: 492, dtype: object

In [16]:
corridors_df.iloc[217]["route_str_str"]

'29238188#0 -25665374 29027727#6 25665375#0 32007121 25901651#3 29027727#9 152077572#0 152077572#2 152077572#2.79 54858974'

In [10]:
len(corridors_df["route_id"].unique())

2134

In [3]:
corridors_df=route_green_time_accessor.get_route_green_time(corridors_df)

In [15]:
corridors_df[["tl","route_id","route_last_edge","route_green_phases","route_green_link_indices","main_green_phase","main_green_phase_duration"]]

,tl,route_id,route_last_edge,route_green_phases,route_green_link_indices,main_green_phase,main_green_phase_duration
0,cluster_277739062_39732718_89129091_89129092,0,-102818928#0.78,[GGGrrrr],"[0, 1]",GGGrrrr,37
1,cluster_277739062_39732718_89129091_89129092,1,-201969576#0,"[rrGGGGg, rryyyyg, rrrrrGG]","[5, 6]",rrGGGGg,38
2,cluster_277739062_39732718_89129091_89129092,2,61940355,"[rrGGGGg, GGGrrrr]","[2, 3, 4]",rrGGGGg,38
3,267380207,3,-32201172,[rrGGrrrGg],"[7, 8]",rrGGrrrGg,27
4,267380207,4,145069373#0,[rrrrGGGrr],"[4, 5, 6]",rrrrGGGrr,27
...,...,...,...,...,...,...,...
781,cluster_475944687_475944688_480162640,781,39649524#0,[GGrrrrr],"[0, 1]",GGrrrrr,16
782,cluster_475944687_475944688_480162640,782,27453586#0,"[rrGGGGr, rryyGGr, rrrrGGG]","[4, 5, 6]",rrGGGGr,40
783,cluster_475944687_475944688_480162640,783,380271165#0,[rrGGGGr],"[2, 3]",rrGGGGr,40
784,2033094557,784,30131201#1,"[GGGrr, GrrrG]","[0, 1]",GGGrr,26


In [7]:
corridors_df.columns

Index(['tl', 'route', 'route_str', 'route_str_str', 'route_last_edge',
       'route_first_edge', 'route_length_ff_tt', 'route_sum_length',
       'route_sum_ff_tt', 'route_ff_speed', 'origin', 'route_n_edges',
       'route_demand', 'tl_demand', 'tl_n_routes', 'tl_n_edges', 'add_info',
       'n_previous_tl', 'n_previous_routes', 'n_previous_edges',
       'previous_demand', 'next_info', 'direct_next_edges', 'direct_next_tls',
       'from_info', 'direct_from_edges', 'direct_previous_tls',
       'direct_next_demand', 'direct_previous_demand',
       'direct_from_n_junction_inc_lanes', 'route_tl_id', 'route_id',
       'route_weight', 'green_phases_info', 'route_green_phases',
       'route_green_link_indices', 'main_green_phase_info', 'main_green_phase',
       'main_green_phase_duration'],
      dtype='str')

In [16]:
df_gt=pd.read_csv("./gt_data/df_gt_InTAS.csv")

In [18]:
df_gt.columns

Index(['Unnamed: 0', 'route_id', 'route_average_traveltime', 'tl', 'route',
       'route_str', 'route_str_str', 'route_last_edge', 'route_first_edge',
       'route_length_ff_tt', 'route_sum_length', 'route_sum_ff_tt',
       'route_ff_speed', 'origin', 'route_n_edges', 'route_demand',
       'tl_demand', 'tl_n_routes', 'tl_n_edges', 'add_info', 'n_previous_tl',
       'n_previous_routes', 'n_previous_edges', 'previous_demand', 'next_info',
       'direct_next_edges', 'direct_next_tls', 'from_info',
       'direct_from_edges', 'direct_previous_tls', 'direct_next_demand',
       'direct_previous_demand', 'direct_from_n_junction_inc_lanes',
       'route_tl_id', 'route_weight'],
      dtype='str')

In [19]:
len(df_gt)

725

In [20]:
len(df_gt["route_id"].unique())

725

In [9]:
additional_gt_xml="./Ingolstadt Network/additional_gt.xml"
net_xml="./Ingolstadt Network/ingolstadt.net.xml"

df_gt=pd.read_csv("./gt_data/df_gt_InTAS.csv")


corridors_df=torch.load("./corridors_savings/corridors_df.pt",weights_only=False)
complete_corridors_df=torch.load("./corridors_savings/complete_corridors_connexions_df.pt",weights_only=False)

initial_tl_df=create_initial_tl_df(additional_gt_xml)

route_green_time_accessor=GetRouteGreenTime(net_file=net_xml,initial_tl_df=initial_tl_df)

df_gt=route_green_time_accessor.get_route_green_time(df_gt)
print(df_gt.shape)
print(len(df_gt["route_id"].unique()))
print(df_gt)
df_gt=pd.merge(df_gt,initial_tl_df[["tl","state","tl_cumulative_constant_duration"]].rename(columns={"state":"main_green_phase"}),on=["tl","main_green_phase"],how="left")
df_gt=df_gt.drop_duplicates(subset=["route_str_str"])
print(df_gt.shape)


(725, 43)
725
     Unnamed: 0  route_id  route_average_traveltime  \
0             0         0                 38.335185   
1             1         1                101.087500   
2             2         2                557.996848   
3             3         3                 60.900000   
4             4         4                148.200000   
..          ...       ...                       ...   
720         720       781                117.547945   
721         721       782                 67.973316   
722         722       783                 19.530253   
723         723       784                 50.992029   
724         724       785                 43.505000   

                                               tl  \
0    cluster_277739062_39732718_89129091_89129092   
1    cluster_277739062_39732718_89129091_89129092   
2    cluster_277739062_39732718_89129091_89129092   
3                                       267380207   
4                                       267380207   
..     

In [10]:
df_gt.columns

Index(['Unnamed: 0', 'route_id', 'route_average_traveltime', 'tl', 'route',
       'route_str', 'route_str_str', 'route_last_edge', 'route_first_edge',
       'route_length_ff_tt', 'route_sum_length', 'route_sum_ff_tt',
       'route_ff_speed', 'origin', 'route_n_edges', 'route_demand',
       'tl_demand', 'tl_n_routes', 'tl_n_edges', 'add_info', 'n_previous_tl',
       'n_previous_routes', 'n_previous_edges', 'previous_demand', 'next_info',
       'direct_next_edges', 'direct_next_tls', 'from_info',
       'direct_from_edges', 'direct_previous_tls', 'direct_next_demand',
       'direct_previous_demand', 'direct_from_n_junction_inc_lanes',
       'route_tl_id', 'route_weight', 'green_phases_info',
       'route_green_phases', 'route_green_link_indices',
       'main_green_phase_info', 'main_green_phase',
       'main_green_phase_duration', 'main_green_phase_to_change',
       'main_green_phase_initial_duration', 'tl_cumulative_constant_duration'],
      dtype='str')

In [2]:
additional_gt_xml="./Ingolstadt Network/additional_gt.xml"
net_xml="./Ingolstadt Network/ingolstadt.net.xml"

df_gt=pd.read_csv("./gt_data/df_gt_InTAS.csv")
initial_tl_df=create_initial_tl_df(additional_gt_xml)
route_green_time_accessor=GetRouteGreenTime(net_file=net_xml,initial_tl_df=initial_tl_df)
x_dim=98
lower_bounds=[5]*x_dim
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=np.array([lower_bounds,upper_bounds])
cdf_inversor=InverseCDF(df_gt=df_gt,initial_tl_df=initial_tl_df,bounds=bounds,route_green_time_accessor=route_green_time_accessor,x_dim=x_dim)

In [3]:
g_vec=cdf_inversor.sample_g_vec()

63.0
46.0
60.0
78.0
60.0
81.0
63.0
69.0
54.0
52.0
65.0
78.0
60.0
78.0
78.0
30.0
78.0
68.0
78.0
30.0
78.0
74.0
69.0
66.0
69.0
56.0
48.0
78.0
48.0
69.0
73.0
59.0
78.0
78.0
69.0
69.0
78.0
69.0
69.0
29.0
30.0
69.0
48.0
69.0
69.0
66.0
66.0
78.0
63.0
78.0
69.0
78.0
60.0
69.0
69.0
52.0
69.0
42.0
52.0
58.0
48.0
42.0
81.0
34.0
69.0
69.0
69.0
48.0
38.0
52.0
60.0
60.0
45.0
69.0
69.0
60.0
69.0
69.0
60.0
42.0
55.0
15.0
69.0
53.0
66.0
69.0
60.0
69.0
42.0
69.0
78.0
63.0
52.0
69.0
63.0
69.0
69.0


In [4]:
g_vec

[59.62274897356842,
 42.669933983564476,
 50.09437107450246,
 18.105264515090234,
 46.88986530519191,
 48.83984410212242,
 58.51506513252903,
 -1,
 45.42152212730357,
 47.78084172614452,
 53.31950112012894,
 53.46186934067556,
 71.39143241678771,
 50.92250156303017,
 35.94732004529713,
 58.14322370953627,
 13.936589278397914,
 30.742986384395362,
 12.404110177379161,
 49.43977119054097,
 6.956259936944803,
 10.547286302997218,
 70.69833151356613,
 72.366585126155,
 70.73006517291175,
 48.54946982325147,
 40.201284223735875,
 7.906178169948131,
 38.106257951901476,
 49.08798120891758,
 23.769226195331825,
 70.79030355585621,
 16.159320393877774,
 44.18289492094241,
 30.983778816040715,
 51.17862372044317,
 20.526593187549246,
 54.01280452209329,
 57.35872113775335,
 8.590897903291612,
 31.070787314876878,
 16.37883496958878,
 36.05584303252131,
 7.985790081630512,
 49.611064126753156,
 45.445087638308365,
 17.405095218324902,
 62.251737838832746,
 54.52266785470475,
 66.44962398154982,
